In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [ ]:
# Create an embedding table for 10 date values
vocab_size = 10
embedding_dim = 16
input_embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)  # 16 is the embedding dimension
  # 3x16
# Example: Initialize the embedding table with random values
print(input_embeddings.weight)

In [ ]:
# Define input embeddings (example tensor)
embedding_dim=16
seq_len, batch_size, embed_dim = 5, 2, embedding_dim  # Use embedding_dim from the embedding table
input_embedding = input_embeddings.weight


# Define linear layers for key, query, and value projections
key_layer = nn.Linear(embed_dim, embed_dim)
query_layer = nn.Linear(embed_dim, embed_dim)
value_layer = nn.Linear(embed_dim, embed_dim)

# Generate key, query, and value tensors
keys = key_layer(input_embedding)
queries = query_layer(input_embedding)
values = value_layer(input_embedding)

print("Keys:", keys)
print("Queries:", queries)
print("Values:", values)

In [ ]:

class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super(CrossAttentionBlock, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout = dropout

        # Multi-head attention for cross-attention
        self.cross_attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Dropout
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, query, key, value, attention_mask=None):
        """
        Forward pass for the cross-attention block.

        Args:
            query: Tensor of shape (seq_len_q, batch_size, embed_dim)
            key: Tensor of shape (seq_len_k, batch_size, embed_dim)
            value: Tensor of shape (seq_len_v, batch_size, embed_dim)
            attention_mask: Optional tensor for masking (seq_len_q, seq_len_k)

        Returns:
            Tensor of shape (seq_len_q, batch_size, embed_dim)
        """
        # Cross-attention
        attn_output, _ = self.cross_attention(query, key, value, attn_mask=attention_mask)
        
        query = query + self.dropout_layer(attn_output)
        print("query:", query)
        query = self.norm1(query)

        # Feed-forward network
        ffn_output = self.ffn(query)
        query = query + self.dropout_layer(ffn_output)
        query = self.norm2(query)

        return query

In [27]:
query = torch.rand(seq_len, batch_size, embed_dim)
key = torch.rand(seq_len, batch_size, embed_dim)
value = torch.rand(seq_len, batch_size, embed_dim)

In [ ]:
num_heads = 8
cross_attn_block = CrossAttentionBlock(embed_dim, num_heads)

In [ ]:
#cheack the cross attention block
# Define the cross-attention block
num_heads = 8


# Generate a random tensor for the query, key, and value
# 5x2x16


# Apply the cross-attention block
# 5x2x16
output = cross_attn_block(query, key, value)
# print(output)


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   
        q = self.query(x) 
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) 
        wei = self.dropout(wei)
        v = self.value(x) 
        out = wei @ v 
        print(out)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

num_heads = 8
cross_attn_block = CrossAttentionBlock(embed_dim, num_heads)

# Generate a random tensor for the query, key, and value
# 5x2x16
# Apply the cross-attention block
# 5x2x16
output = cross_attn_block(query, key, value)


    

In [ ]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, dropout=0.1):
        super(Transformer, self).__init__()
        self.layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query, key, value, attention_mask=None):
        """
        Forward pass for the transformer.

        Args:
            query: Tensor of shape (seq_len_q, batch_size, embed_dim)
            key: Tensor of shape (seq_len_k, batch_size, embed_dim)
            value: Tensor of shape (seq_len_v, batch_size, embed_dim)
            attention_mask: Optional tensor for masking (seq_len_q, seq_len_k)

        Returns:
            Tensor of shape (seq_len_q, batch_size, embed_dim)
        """
        for layer in self.layers:
            query = layer(query, key, value, attention_mask)
        return self.norm(query)

# Example usage
num_layers = 6
transformer = Transformer(embed_dim, num_heads, num_layers)

# Apply the transformer
output = transformer(query, key, value)
print(output)